In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import FloatSlider, IntSlider, VBox, HBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:1050px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#12388c;
    margin-bottom:8px;
">
Spectral Factorization: From a Desired PSD to an LTI System
</div>

<div style="margin-bottom:4px;">
<b>Goal:</b> construct a causal and stable LTI system whose output has a prescribed power spectral density.
</div>

<div style="margin-bottom:4px;">
For white-noise input, the output spectrum satisfies <b>Sₓₓ(ω) = |H(eʲω)|²</b>.
</div>

<div style="margin-bottom:4px;">
Spectral factorization assigns the poles inside the unit circle to the stable factor H(z).
</div>

<div>
<b>This notebook:</b> compares the desired PSD with the PSD obtained numerically by filtering white noise.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}

slider_layout = Layout(width='155px')

p_slider = FloatSlider(
    min=0.10,
    max=0.90,
    step=0.05,
    value=0.70,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

q_slider = FloatSlider(
    min=0.10,
    max=0.90,
    step=0.05,
    value=0.40,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

K_slider = FloatSlider(
    min=0.20,
    max=3.00,
    step=0.10,
    value=1.00,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

N_slider = IntSlider(
    min=1024,
    max=8192,
    step=1024,
    value=4096,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

p_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.70</div>')

q_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.40</div>')

K_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>')

N_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">4096</div>')

# ============================================================
# UPDATE CURRENT VALUES
# ============================================================

def update_p_value(change):
    p_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{p_slider.value:.2f}</div>'

def update_q_value(change):
    q_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{q_slider.value:.2f}</div>'

def update_K_value(change):
    K_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{K_slider.value:.2f}</div>'

def update_N_value(change):
    N_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{N_slider.value}</div>'

p_slider.observe(update_p_value, names='value')

q_slider.observe(update_q_value, names='value')

K_slider.observe(update_K_value, names='value')

N_slider.observe(update_N_value, names='value')

# ============================================================
# CONTROL LABELS
# ============================================================

p_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Pole p:</div>')

q_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Pole q:</div>')

K_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">PSD scale K:</div>')

N_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold;">Samples N:</div>')

# ============================================================
# CONTROLS GRID
# ============================================================

controls_grid = GridBox(
    children=[
        p_label, p_slider, p_value,
        q_label, q_slider, q_value,
        K_label, K_slider, K_value,
        N_label, N_slider, N_value
    ],
    layout=Layout(
        width='345px',
        grid_template_columns='100px 155px 55px',
        grid_template_rows='34px 34px 34px 34px',
        grid_gap='4px 6px',
        align_items='center',
        overflow='hidden'
    )
)

# ============================================================
# PARAMETERS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#12388c;
            margin-bottom:7px;
        ">
        Parameters
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='370px',
        min_width='370px',
        padding='12px 12px',
        border='1px solid #d2d2d2',
        overflow='hidden',
        margin='18px 0px 0px 12px'
    )
)

# ============================================================
# GRAPH 1:
# SPECTRAL FACTORIZATION IN THE z-PLANE
# ============================================================

def plot_zplane(p=0.70, q=0.40):

    theta = np.linspace(0.0, 2.0 * np.pi, 500)

    inside_poles = np.array([p, q])

    outside_poles = np.array([1.0 / p, 1.0 / q])

    fig, ax = plt.subplots(figsize=(6.6, 4.4))

    ax.plot(
        np.cos(theta),
        np.sin(theta),
        linestyle='--',
        linewidth=1.3,
        label='Unit circle'
    )

    ax.axhline(
        0.0,
        linewidth=0.8
    )

    ax.axvline(
        0.0,
        linewidth=0.8
    )

    ax.scatter(
        inside_poles,
        np.zeros_like(inside_poles),
        marker='o',
        s=110,
        facecolors='none',
        linewidths=2.0,
        label='Poles selected for H(z)'
    )

    ax.scatter(
        outside_poles,
        np.zeros_like(outside_poles),
        marker='x',
        s=90,
        linewidths=2.0,
        label='Reciprocal PSD poles'
    )

    horizontal_limit = max(
        1.4,
        1.15 / min(p, q)
    )

    horizontal_limit = min(
        horizontal_limit,
        11.0
    )

    ax.set_xlim(
        -0.5,
        horizontal_limit
    )

    ax.set_ylim(
        -1.3,
        1.3
    )

    ax.set_xlabel(
        'Real part',
        fontsize=11
    )

    ax.set_ylabel(
        'Imaginary part',
        fontsize=11
    )

    ax.set_title(
        'Spectral Factorization in the z-Plane',
        fontsize=13,
        pad=10
    )

    ax.tick_params(
        axis='both',
        labelsize=9
    )

    ax.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.17),
        ncol=2,
        fontsize=8
    )

    fig.subplots_adjust(
        left=0.11,
        right=0.97,
        top=0.88,
        bottom=0.25
    )

    plt.show()

    plt.close(fig)

# ============================================================
# GRAPH 2:
# DESIRED PSD VS GENERATED OUTPUT PSD
# ============================================================

def plot_psd_comparison(p=0.70, q=0.40, K=1.00, N=4096):

    # --------------------------------------------------------
    # FREQUENCY AXIS
    # --------------------------------------------------------

    omega = np.linspace(
        -np.pi,
        np.pi,
        2048
    )

    # --------------------------------------------------------
    # CAUSAL / STABLE SPECTRAL FACTOR
    # --------------------------------------------------------

    H = np.sqrt(K) / (
        (1.0 - p * np.exp(-1j * omega))
        *
        (1.0 - q * np.exp(-1j * omega))
    )

    # --------------------------------------------------------
    # DESIRED PSD
    # --------------------------------------------------------

    S_target = np.abs(H) ** 2

    # --------------------------------------------------------
    # WHITE-NOISE INPUT
    # --------------------------------------------------------

    rng = np.random.default_rng(10)

    w = rng.normal(
        0.0,
        1.0,
        N
    )

    # --------------------------------------------------------
    # FILTER COEFFICIENTS
    # --------------------------------------------------------

    b = np.array(
        [
            np.sqrt(K)
        ]
    )

    a = np.array(
        [
            1.0,
            -(p + q),
            p * q
        ]
    )

    # --------------------------------------------------------
    # FILTER WHITE NOISE
    # --------------------------------------------------------

    x = signal.lfilter(
        b,
        a,
        w
    )

    transient = min(
        500,
        N // 8
    )

    x_ss = x[transient:]

    # --------------------------------------------------------
    # ESTIMATED PSD
    # --------------------------------------------------------

    nperseg = min(
        512,
        len(x_ss)
    )

    f_est, Pxx = signal.welch(
        x_ss,
        fs=2.0 * np.pi,
        window='hann',
        nperseg=nperseg,
        noverlap=nperseg // 2,
        return_onesided=False,
        scaling='density'
    )

    f_est = np.fft.fftshift(
        f_est
    )

    Pxx = np.fft.fftshift(
        Pxx
    )

    Pxx = 2.0 * np.pi * Pxx

    # --------------------------------------------------------
    # FIGURE
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(8.8, 4.3)
    )

    ax.plot(
        omega,
        S_target,
        linewidth=2.2,
        label='Desired theoretical PSD'
    )

    ax.plot(
        f_est,
        Pxx,
        linewidth=1.2,
        alpha=0.75,
        label='Estimated output PSD'
    )

    ax.set_xlim(
        -np.pi,
        np.pi
    )

    ax.set_xticks(
        [
            -np.pi,
            -np.pi / 2.0,
            0.0,
            np.pi / 2.0,
            np.pi
        ]
    )

    ax.set_xticklabels(
        [
            '-π',
            '-π/2',
            '0',
            'π/2',
            'π'
        ]
    )

    ax.set_xlabel(
        'Angular frequency ω',
        fontsize=11
    )

    ax.set_ylabel(
        'Power spectral density',
        fontsize=11
    )

    ax.set_title(
        'Desired PSD and Generated Output PSD',
        fontsize=13,
        pad=10
    )

    ax.tick_params(
        axis='both',
        labelsize=9
    )

    ax.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.19),
        ncol=2,
        fontsize=8
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.97,
        top=0.88,
        bottom=0.25
    )

    plt.show()

    plt.close(fig)

# ============================================================
# CURRENT FACTORIZATION
# ============================================================

factorization_output = HTML()

def update_factorization(p=0.70, q=0.40, K=1.00):

    denominator_1 = p + q

    denominator_2 = p * q

    factorization_output.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.40;
        width:880px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Current spectral factor:</b>
    &nbsp;&nbsp;
    H(z) = {np.sqrt(K):.3f} /
    [1 - {denominator_1:.3f} z<sup>-1</sup>
    + {denominator_2:.3f} z<sup>-2</sup>]

    <br>

    <b>Selected stable poles:</b>
    p = {p:.2f},
    q = {q:.2f}

    &nbsp;&nbsp;&nbsp;

    <b>Reciprocal poles:</b>
    1/p = {1.0/p:.3f},
    1/q = {1.0/q:.3f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUTS
# ============================================================

zplane_output = interactive_output(
    plot_zplane,
    {
        'p': p_slider,
        'q': q_slider
    }
)

psd_output = interactive_output(
    plot_psd_comparison,
    {
        'p': p_slider,
        'q': q_slider,
        'K': K_slider,
        'N': N_slider
    }
)

factorization_interactive = interactive_output(
    update_factorization,
    {
        'p': p_slider,
        'q': q_slider,
        'K': K_slider
    }
)

# ============================================================
# FIRST ROW:
# GRAPH LEFT - PARAMETERS RIGHT
# ============================================================

first_row = HBox(
    [
        zplane_output,
        controls_card
    ],
    layout=Layout(
        width='1050px',
        align_items='center',
        justify_content='flex-start',
        overflow='hidden',
        margin='4px 0px 0px 0px'
    )
)

# ============================================================
# SHORT INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.42;
    width:1050px;
    padding:11px 15px;
    border:1px solid #c8dfce;
    background:#f8fcf9;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#197b35;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The PSD contains reciprocal pole pairs; the poles inside the unit circle are selected for the causal and stable factor H(z).
</div>

<div style="margin-bottom:4px;">
Filtering unit-variance white noise with H(z) produces a random sequence whose theoretical PSD is |H(eʲω)|².
</div>

<div>
The estimated PSD fluctuates around the theoretical curve because it is obtained from a finite random realization.
</div>

</div>
""")

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        documentation,
        first_row,
        psd_output,
        factorization_interactive,
        factorization_output,
        interpretation
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)